# Metrics stage tour
A hands-on tour of the Metrics stage (`galactic_tidal_constraints.simr.metrics`, MDRN-4 step 2a): one
function per metric, computing a value from simulated or external arrays. Scalar methods return one float;
descriptive methods return a dict of declared keys. Run the cells in order.

`simr` doesn't run metrics yet (step 2b adds the `sim_metrics` / `img_metrics` config sections), so here
each method is called directly on real arrays: an orbit from `sim`, and images made through `simr`.

In [ ]:
%matplotlib inline
import json

import matplotlib.pyplot as plt
import numpy as np

from galactic_tidal_constraints.simr import metrics
from galactic_tidal_constraints.simr import simr
from galactic_tidal_constraints.simr import simulation as sim

# Model SDSS 587722984435351614 (as in the other tours), and a copy with a heavier secondary.
params = np.array(
    [-9.93853, -4.5805, 15.43348, -1.54025, -2.97899, 4.69234, 34.41671, 29.49607, 2.97913, 4.2501,
     301.0, 35.5, 310.80745, 321.98757]
)
heavier = params.copy()
heavier[sim.PARAM_INDEX["mass2"]] *= 2

## The metrics table
Each method is registered with the `metric` decorator where it's defined, which checks the declaration
at import and records it in `metrics.METHODS` — one source of truth. `metrics.table()` is the same
information as plain data: kind, input kinds, settings and defaults, direction (`best`) and `range` for
scalars, output keys and shapes for descriptors (`[]` is one float).

In [ ]:
print(json.dumps(metrics.table(), indent=2))

## `closest_approach`: a descriptive metric
`sim.search_orbit` is the secondary's center from now back to t = −30 (the path SPAM integrates backward
to place a run's start; a `simr` item since step 1). `closest_approach` finds the point along whatever
orbit it's given where the two centers are closest, and returns everything about it in one dict.

In [ ]:
path = sim.search_orbit(params).astype(np.float32)  # as the simr item hands it out
approach = metrics.closest_approach(path)
approach

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(path[:, 0], path[:, 1], lw=0.8, label="search orbit (t = 0 back to \u221230)")
ax.plot(0, 0, "k+", ms=12, label="primary (origin)")
ax.plot(path[0, 0], path[0, 1], "s", color="tab:green", label="now (t = 0)")
ax.plot(*approach["position"][:2], "o", color="tab:red",
        label=f"closest: d = {approach['distance']:.2f} at t = {approach['t']:.2f}")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_aspect("equal")
ax.legend()
ax.set_title("closest_approach on sim.search_orbit")
plt.show()

## The entry's input decides the window
The method itself has no window: it reports the smallest distance in whatever orbit it's given. On
`sim.search_orbit` that's the whole window back to t = −30; on `sim.orbit` it's the run itself, so the
simulator config's `start` decides. Here the run starts before the most recent pass, so both find the
same pass; a run started later, or a window with an older pass, would differ.

In [ ]:
run_orbit = sim.simulate_orbit(params).astype(np.float32)
print("sim.search_orbit:", {k: approach[k] for k in ("t", "distance")})
print("sim.orbit:       ", {k: metrics.closest_approach(run_orbit)[k] for k in ("t", "distance")})

## `select`: lifting one float out of a descriptor
Composites (step 5) take scalars only, so a config entry can run a descriptive method and keep one
single-float key. Only keys with shape `()` qualify (`Metric.scalar_outputs`); naming an array-valued
key like `position` will be a config error caught before anything runs. In step 2b an entry will look like:
```json
"sim_metrics": [
  {"method": "closest_approach", "inputs": ["sim.search_orbit"]},
  {"method": "closest_approach", "inputs": ["sim.search_orbit"], "select": "distance"}
]
```

In [ ]:
print("scalar_outputs:", metrics.METHODS["closest_approach"].scalar_outputs)
print("select 'distance':", approach["distance"])
print("select 't':       ", approach["t"])

## `absolute_difference`: a scalar metric on two images
The mean |a − b| over the pixels of two greyscale images of the same shape: 0 is identical, and any two
matching images work, wherever they come from. Model vs unperturbed reads as how perturbed the model is;
against an external target (step 3) it will read as how far from the target. Images already flow through
`simr`, so both come from one run (one default image entry; `data` is indexed `[model, simulator, image]`).

In [ ]:
result = simr.run(np.stack([params, heavier]), {"run": {"return": ["img.model", "img.unperturbed"]}})
differences = [
    metrics.absolute_difference(result.data["img.model"][m, 0, 0], result.data["img.unperturbed"][m, 0, 0])
    for m in range(2)
]
differences

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(8, 8))
for m, name in enumerate(["model", "heavier secondary"]):
    axes[m, 0].imshow(result.data["img.model"][m, 0, 0], cmap="gray")
    axes[m, 0].set_title(f"{name}: img.model")
    axes[m, 1].imshow(result.data["img.unperturbed"][m, 0, 0], cmap="gray")
    axes[m, 1].set_title(f"img.unperturbed \u2192 difference {differences[m]:.4f}")
for ax in axes.flat:
    ax.set_axis_off()
plt.tight_layout()
plt.show()

## Mismatched shapes are an error
Matching image frames is the caller's job (no resize or shift inside the stage): two images of different
shapes are an error, not a silent crop. Once externals exist (step 3), this is caught before anything
runs, through the early shape check.

In [ ]:
try:
    metrics.absolute_difference(np.zeros((128, 128), np.float32), np.zeros((64, 64), np.float32))
except ValueError as error:
    print("ValueError:", error)

## Metrics through `simr` (step 2b)
The `sim_metrics` and `img_metrics` config sections list entries (`method`, `inputs`, optional `select`
and `params`), checked against the metrics table before anything runs. Every `sim_metrics` entry runs on
every simulation and every `img_metrics` entry on every (simulation, image) pair, and the values always
come back: `result.sim_metrics[model, simulator, entry]` and `result.img_metrics[model, simulator, image,
entry]`, with per-cell status, time and error. `simr` computes what the entries need even when nothing
else asks for it — this request returns no data at all, only metrics.

In [ ]:
request = {
    "sim_metrics": [
        {"method": "closest_approach", "inputs": ["sim.search_orbit"]},
        {"method": "closest_approach", "inputs": ["sim.search_orbit"], "select": "distance"},
    ],
    "img_metrics": [{"method": "absolute_difference", "inputs": ["img.model", "img.unperturbed"]}],
}
metric_run = simr.run(np.stack([params, heavier]), request)
print("data returned:", metric_run.data)
print("sim_metrics", metric_run.sim_metrics.shape, "| img_metrics", metric_run.img_metrics.shape)
print("closest approach of model 0:", metric_run.sim_metrics[0, 0, 0])
print("select 'distance' per model:", metric_run.sim_metrics[:, 0, 1])
print("image difference per model: ", metric_run.img_metrics[:, 0, 0, 0])
print("filled-in entry 1:", metric_run.sim_metric_configs[1])
print("status:", metric_run.status["sim_metrics"].tolist())

## PCA: a metric for each particle input
`pca_shape` describes a particle cloud by its covariance ellipsoid (3D principal component analysis), on
the whole array as given: for `sim.particles` that's both galaxies together. `axes` are the semi-axes
(standard deviation along each principal axis), largest first; `vectors` the matching unit vectors as
rows, each with its largest component positive so the output is deterministic; `b_a` and `c_a` the axis
ratios (single floats, so `select` can pick them). Settings: `columns` (`"position"` or `"velocity"`),
`center` (`"mean"` or `"median"`) and `trim_quantile` (off by default: tidal tails are the outliers).

The plots show each cloud in x–y with its three axes at ±2σ, projected.

In [ ]:
# A different seed for each model, so their disks don't start from the same particle pattern: two
# simulator entries (seeds 1 and 2), taking model m from entry m. data is [model, simulator].
request = {"simulator": [{"seed": 1}, {"seed": 2}], "run": {"return": ["sim.particles", "sim.unperturbed"]}}
particle_run = simr.run(np.stack([params, heavier]), request)
clouds = [{item: particle_run.data[item][m, m] for item in ("sim.particles", "sim.unperturbed")} for m in range(2)]
shapes = [{item: metrics.pca_shape(cloud) for item, cloud in pair.items()} for pair in clouds]
print("shape:", clouds[0]["sim.particles"].shape)
shapes[0]["sim.particles"]

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(9, 9))
for m, model in enumerate(["model", "heavier secondary"]):
    for k, item in enumerate(["sim.particles", "sim.unperturbed"]):
        ax, cloud, shape = axes[m, k], clouds[m][item], shapes[m][item]
        ax.scatter(cloud[:, 0], cloud[:, 1], s=0.2, color="0.4")
        for axis, vector, color in zip(shape["axes"], shape["vectors"], ["tab:red", "tab:blue", "tab:green"]):
            ends = shape["center"][:2] + np.outer([-1, 1], 2 * axis * vector[:2])  # +-2 sigma
            ax.plot(ends[:, 0], ends[:, 1], color=color, lw=2)
        ax.set_title(f"{model}: {item}\nb/a {shape['b_a']:.2f}, c/a {shape['c_a']:.2f}")
        ax.set_aspect("equal")
plt.tight_layout()
plt.show()

### `pca_difference`: a scalar metric on two particle clouds
How different two clouds' shapes are, from their ellipsoids: 0 is the same shape, and the clouds may
hold different numbers of particles. The default compares the axis ratios `[b/a, c/a]` with an L2
distance, blind to size, position and rotation (values in [0, √2]). `descriptor="axes"` lets size count,
`"geometric"` compares `[linearity, planarity, sphericity]`, and `orientation_weight` w adds
w·(1 − |v1_a·v1_b|) for how far apart the longest axes point (SPAM's frame is the sky's, so orientation
is real information). Model vs unperturbed reads as how much the encounter changed the system's shape.

In [ ]:
for name, settings in [
    ("ratios, l2 (default)", {}),
    ("ratios, l1", {"distance": "l1"}),
    ("axes (size counts)", {"descriptor": "axes"}),
    ("geometric", {"descriptor": "geometric"}),
    ("ratios + orientation (w = 1)", {"orientation_weight": 1.0}),
    ("velocities", {"columns": "velocity"}),
]:
    values = [metrics.pca_difference(clouds[m]["sim.particles"], clouds[m]["sim.unperturbed"], **settings)
              for m in range(2)]
    print(f"{name:30s} model {values[0]:.4f}   heavier {values[1]:.4f}")

### All four particle sets against each other
Every pair (rows vs columns), one table per descriptor: 0 on the diagonal, and the off-diagonal cells show
which sets have similar shapes.

In [ ]:
sets = {
    "model / perturbed": clouds[0]["sim.particles"],
    "model / unperturbed": clouds[0]["sim.unperturbed"],
    "heavier / perturbed": clouds[1]["sim.particles"],
    "heavier / unperturbed": clouds[1]["sim.unperturbed"],
}
names = list(sets)
for descriptor in metrics.PCA_DESCRIPTORS:
    grid = np.array([[metrics.pca_difference(sets[a], sets[b], descriptor=descriptor) for b in names] for a in names])
    print(f"descriptor={descriptor!r} (rows vs columns, 0 = same shape)")
    print(" " * 24 + "".join(f"{n:>24s}" for n in names))
    for name, row in zip(names, grid):
        print(f"{name:24s}" + "".join(f"{v:24.4f}" for v in row))
    print()

### PCA metrics through `simr`
Like any method, the PCA pair runs from config entries; settings go in `params`, and the filled-in entry
records every setting with its default.

In [ ]:
request = {
    "sim_metrics": [
        {"method": "pca_shape", "inputs": ["sim.particles"], "select": "c_a"},
        {"method": "pca_difference", "inputs": ["sim.particles", "sim.unperturbed"]},
        {"method": "pca_difference", "inputs": ["sim.particles", "sim.unperturbed"],
         "params": {"orientation_weight": 1.0}},
    ],
}
pca_run = simr.run(np.stack([params, heavier]), request)
for e, entry in enumerate(pca_run.sim_metric_configs):
    print(entry["method"], entry["params"], "->", pca_run.sim_metrics[:, 0, e].tolist())

## What's next
Step 3 adds externals (targets and other named arrays as metric inputs), step 4 the orbit metrics
(the raised-cosine weight, time and position scores against a target orbit), step 5 the composite,
and step 7 saves metric values to the catalog and `model.json` (today they only come back in memory;
the command line warns about that).